# leg_RAG — Eval Question Workbench

A hands-on tool for building the 75-100 question eval set: browse real
cases, read them in full, and save your drafted questions as you go — this
notebook writes to `eval/labeled_set_draft.jsonl` immediately as you add
each one, so nothing is lost between sessions.

See `eval/LABELING_GUIDE.md` for the full reading guide and glossary — this
notebook is the hands-on companion to it.

Run from the project root so `import config` resolves — this notebook
finds it automatically by walking upward, so it's safe to move.

In [ ]:
import csv
import json
import sys
import textwrap
from pathlib import Path
import numpy as np
import pandas as pd

_search = Path.cwd()
while not (_search / "config.py").exists():
    if _search.parent == _search:
        raise RuntimeError("Could not find project root (config.py) — open this notebook from within the leg_RAG project.")
    _search = _search.parent
PROJECT_ROOT = _search
sys.path.insert(0, str(PROJECT_ROOT))
import config
assert hasattr(config, "PROCESSED_DIR"), "Imported the wrong 'config' module — check for a stray pip package named 'config'."

csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

from eval.candidate_picker import pick_candidates, _extract_preview

DRAFT_PATH = config.ROOT / "eval" / "labeled_set_draft.jsonl"


## 1. What the raw data actually looks like

Before browsing cases, here's one real row from each of the two files that
matter — so "opinion" and "cluster" stop being abstract words.

In [2]:
with open(config.PROCESSED_DIR / "opinions_scoped_loose.csv", encoding="utf-8", newline="") as f:
    one_opinion = next(csv.DictReader(f))

print("--- One row from opinions_scoped_loose.csv (the TEXT file) ---")
for k, v in one_opinion.items():
    v_display = (v[:80] + "...") if isinstance(v, str) and len(v) > 80 else v
    print(f"  {k}: {v_display!r}")

--- One row from opinions_scoped_loose.csv (the TEXT file) ---
  id: '4104545'
  date_created: '2016-12-05 21:01:09.751648+00'
  date_modified: '2025-07-23 14:00:36.582891+00'
  author_str: ''
  per_curiam: 'f'
  joined_by_str: ''
  type: '010combined'
  sha1: '773383be6ec6f7dfcbb168237b8cb94966253a5a'
  page_count: '6'
  download_url: 'http://cdn.ca9.uscourts.gov/datastore/memoranda/2016/12/05/15-55313.pdf'
  local_path: 'pdf/2016/12/05/leon_thomas_v._francisco_quintana.pdf'
  plain_text: '                                                                            FILE...'
  html: ''
  html_lawbox: ''
  html_columbia: ''
  html_anon_2020: ''
  xml_harvard: '<?xml version="1.0" encoding="utf-8"?>\n<opinion type="majority">\n<p id="b690-9">...'
  xml_scan: ''
  html_with_citations: '<?xml version="1.0" encoding="utf-8"?>\n<opinion type="majority">\n<p id="b690-9">...'
  extracted_by_ocr: 'f'
  author_id: ''
  cluster_id: '4327284'


In [3]:
cluster_ids_needed = {one_opinion["cluster_id"]}
with open(config.PROCESSED_DIR / "opinion_clusters_scoped.csv", encoding="utf-8", newline="") as f:
    one_cluster = next(row for row in csv.DictReader(f) if row["id"] in cluster_ids_needed)

print("--- The matching row from opinion_clusters_scoped.csv (the LABEL file) ---")
for k, v in one_cluster.items():
    v_display = (v[:80] + "...") if isinstance(v, str) and len(v) > 80 else v
    print(f"  {k}: {v_display!r}")

print()
print("Notice: opinions_scoped_loose.csv has the 'cluster_id' column — that's the")
print("link between the two files. You'll never need to do this lookup by hand;")
print("the helper functions below do it for you.")

--- The matching row from opinion_clusters_scoped.csv (the LABEL file) ---
  id: '4327284'
  date_created: '2016-12-05 21:01:09.744151+00'
  date_modified: '2024-11-06 06:25:11.862323+00'
  judges: 'Berzon, Nguyen, Zouhary'
  date_filed: '2016-12-05'
  date_filed_is_approximate: 'f'
  slug: 'leon-thomas-v-francisco-quintana'
  case_name_short: ''
  case_name: 'Leon Thomas v. Francisco Quintana'
  case_name_full: 'Leon THOMAS, Plaintiff-Appellee, v. Francisco QUINTANA; And Bradley Jurgensen, D...'
  scdb_id: ''
  scdb_decision_direction: ''
  scdb_votes_majority: ''
  scdb_votes_minority: ''
  source: 'CU'
  procedural_history: ''
  attorneys: 'David H. Kwasniewski, Steptoe & Johnson LLP, Palo Alto, CA, for Plaintiff-Ap-pel...'
  nature_of_suit: 'Prisoner'
  posture: ''
  syllabus: ''
  headnotes: ''
  summary: ''
  disposition: ''
  history: ''
  other_dates: 'Argued and Submitted November 10, 2016 Pasadena, California'
  cross_reference: ''
  correction: ''
  citation_count: '1'
  pre

## 2. Quick term reference

- **Opinion** = the actual written court document (what you read). One row in the text file.
- **Cluster** = the case-level record (name, date, published status) that opinion belongs to.
- **`opinion_id`** = the ID you'll use to ask for a case (e.g. in `read_case(...)` below).
- **Published vs Unpublished** = Published carries more legal weight; Unpublished is still real, just less binding.

Full glossary (qualified immunity, clearly established, holding vs. dicta, etc.) is in `eval/LABELING_GUIDE.md`.

## 3. Load the lookup helpers

Builds two dictionaries in memory — `opinions` (by `opinion_id`) and
`clusters` (by `cluster_id`) — so the rest of this notebook can look up any
case instantly.

In [4]:
with open(config.PROCESSED_DIR / "opinions_scoped_loose.csv", encoding="utf-8", newline="") as f:
    opinions = {row["id"]: row for row in csv.DictReader(f)}

needed_cluster_ids = {o["cluster_id"] for o in opinions.values()}
with open(config.PROCESSED_DIR / "opinion_clusters_scoped.csv", encoding="utf-8", newline="") as f:
    clusters = {row["id"]: row for row in csv.DictReader(f) if row["id"] in needed_cluster_ids}

print(f"{len(opinions)} opinions and {len(clusters)} matching clusters loaded and ready to browse.")


def case_name(opinion_id):
    return clusters.get(opinions[opinion_id]["cluster_id"], {}).get("case_name", "")


def case_date(opinion_id):
    return clusters.get(opinions[opinion_id]["cluster_id"], {}).get("date_filed", "")


def case_status(opinion_id):
    return clusters.get(opinions[opinion_id]["cluster_id"], {}).get("precedential_status", "")


def read_case(opinion_id, wrap_width=100):
    """Prints one case's full text, wrapped for readability."""
    o = opinions[opinion_id]
    print(f"{case_name(opinion_id)}  ({case_date(opinion_id)})  [{case_status(opinion_id)}]  [opinion_id={opinion_id}]")
    print("=" * wrap_width)
    for paragraph in o["plain_text"].split(chr(10)):
        if paragraph.strip():
            print(textwrap.fill(paragraph, wrap_width))
        else:
            print()

1696 opinions and 1696 matching clusters loaded and ready to browse.


## 4. Get a batch of candidate cases to skim

Change the second number (the seed) to get a completely different random
batch — try `43`, `44`, etc. Published-first, but Unpublished included too
(see `candidate_picker.py`'s docstring for why).

In [5]:
BATCH_SEED = 42  # change this for a fresh batch
candidates = pick_candidates(n=15, seed=BATCH_SEED)

rows = []
for o in candidates:
    _, conclusion = _extract_preview(o["plain_text"])
    rows.append({
        "opinion_id": o["id"],
        "case_name": o["case_name"],
        "date": o["date_filed"],
        "status": o["precedential_status"],
        "likely_conclusion_preview": conclusion[-200:].replace(chr(10), " "),
    })

pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

,opinion_id,case_name,date,status,likely_conclusion_preview
0,6347984,Linda Senn v. Kyle Smith,2022-06-08,Published,ef through a consent decree or settlement. Whatever relief the plaintiff sec...
1,3065243,Joseph Padgett v. Brian Loventhal,2009-11-20,Published,quest made in an appellate brief does not satisfy Rule 38 . . . .” (quoting ...
2,4390937,Marty Emmons v. City of Escondido,2019-04-25,Published,e Court’s demand for specificity. Officer Craig is therefore entitled to qua...
3,3064545,Maropulos v. County of Los Angeles,2009-03-24,Published,"ualified immunity and, when it is for reasons of sufficiency of the evidence..."
4,3051877,Torresl v. City of Madera,2008-05-05,Published,"onable under Graham, 490 U.S. at 396-97, and to otherwise proceed with the m..."
5,858125,Donald Wige v. City of Los Angeles,2013-04-16,Published,whether Officer Bellows should be believed. That is the issue Wige seeks to ...
6,4579680,Maria Ventura v. Jennifer Rutledge,2020-10-22,Published,particular conduct was unlawful”). Officer Rutledge is entitled to qua...
7,3031371,Baldwin v. Placer County,2005-04-18,Published,lation of an established constitutional right as the perjury itself. Whether...
8,3040365,Adams v. Speers,2007-01-10,Published,ence of warning and the lack of danger to the shooter or others distinguish ...
9,175426,Braswell v. Shoreline Fire Department,2010-09-16,Published,"er proceedings.2 As to the remain- ing claims, we affirm. AFFIRMED in par..."


## 5. Read one case in full

Copy an `opinion_id` from the table above (or from anywhere earlier in this
notebook) into the line below, then run the cell.

In [6]:
OPINION_ID = "6347984"  # <- change this

read_case(OPINION_ID)

Linda Senn v. Kyle Smith  (2022-06-08)  [Published]  [opinion_id=6347984]
               FOR PUBLICATION

 UNITED STATES COURT OF APPEALS
      FOR THE NINTH CIRCUIT


LINDA SENN,                               No. 21-35293
                 Plaintiff-Appellee,
                                             D.C. No.
                v.                        3:18-cv-01814-
                                                HZ
KYLE SMITH,
              Defendant-Appellant,
                                             ORDER
               and

CITY OF PORTLAND; LARRY
GRAHAM; JEFFREY MCDANIEL;
MULTNOMAH COUNTY; JOHN DOES,
1–10,
                     Defendants.

                     Filed June 8, 2022

 Before: Susan P. Graber, Carlos T. Bea, and Milan D.
              Smith, Jr., Circuit Judges.

                           Order
 2                         SENN V. SMITH

                          SUMMARY *


                 Civil Rights/Attorneys’ Fees

    The panel denied a motion for attorney’

## 6. Draft a question

Fill in the fields and run this cell — it saves immediately to
`eval/labeled_set_draft.jsonl`, so your work persists even if you close the
notebook. Re-running with the exact same question text won't create a
duplicate. Copy this cell (Jupyter: select it, press `b` to insert a new
one below) for each new question — don't overwrite this one, keep it as a
template.

In [7]:
def add_question(question, answer, supporting_cases, qtype="direct", trap_details=None):
    """supporting_cases: list of {"case_name", "opinion_id", "passage"}.
    qtype: "direct", "multi_hop", "trap", or "no_answer"."""
    existing = []
    if DRAFT_PATH.exists():
        with open(DRAFT_PATH, encoding="utf-8") as f:
            existing = [json.loads(line) for line in f]

    if any(e["question"] == question for e in existing):
        print(f"Already saved (skipped duplicate): {question}")
        return

    entry = {"question": question, "answer": answer, "supporting_cases": supporting_cases, "type": qtype}
    if trap_details:
        entry["trap_details"] = trap_details

    with open(DRAFT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"Saved. Total questions so far: {len(existing) + 1}")


# Worked example (the Senn v. Smith case from earlier) — copy this pattern below.
add_question(
    question="If a plaintiff successfully defeats an officer's qualified immunity defense but the case hasn't gone to trial yet, are they entitled to attorney's fees?",
    answer="No. In Senn v. Smith (9th Cir. 2022), the court held that defeating a motion for qualified immunity only earns the plaintiff the right to a trial — it does not make them a 'prevailing party' under 42 U.S.C. section 1988(b), so attorney's fees are not yet available at that stage.",
    supporting_cases=[{
        "case_name": "Senn v. Smith",
        "opinion_id": "6347984",
        "passage": "We deny fees because Plaintiff is not a 'prevailing party' within the meaning of Section 1988(b)... because the plaintiff has not yet prevailed on any claim.",
    }],
    qtype="direct",
)

Saved. Total questions so far: 1


## 7. Your progress so far

Re-run this cell anytime to see everything you've saved.

In [8]:
if DRAFT_PATH.exists():
    with open(DRAFT_PATH, encoding="utf-8") as f:
        drafted = [json.loads(line) for line in f]
    print(f"{len(drafted)} / ~90 questions drafted so far\n")
    pd.DataFrame(drafted)[["question", "type"]]
else:
    print("No questions saved yet — run section 6 to add your first one.")

1 / ~90 questions drafted so far

